# Lesson 01 — 给桌面提醒器装一个按钮

你已经会一些 C++；这课不从语法课开始。我们先用一个可运行的 C++ 控制器解决按钮问题，然后把它弄坏，再通过输出和测试找出原因。

**使用方式：文字 → C++ 代码格 → 下方可操作的虚拟板卡 → 修改 → 再运行。** 不需要 Python 基础。

## 第一次打开

在 VS Code 中打开本项目文件夹，再打开此 Notebook。右上角 **Select Kernel / 选择内核 → Python Environments → `.venv`**。这个 kernel 只是运行支架；你写的实验代码仍是 C++。

按格左侧的 ▶ 或 `Shift+Enter` 运行。先从初始化格开始。如果第一次构建稍慢，等待它完成。首次显示控件时，VS Code 若提示下载 `@jupyter-widgets/base` / `controls`，选择 **启用下载**，加载标准 Jupyter widget 前端资源。


In [ ]:
# 运行支架：只运行一次即可，不是 Python 学习内容，不需要修改。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


## 0. 先看看：ESP32 到底是什么？

先运行下一格，切换部件下拉框，认识模组、开发板、供电和信号针脚。我们以 **经典 ESP32 + ESP32-DevKitC V4** 为参考；ESP32 是一个家族，不是所有板子的针脚都相同。

GPIO 数字是芯片信号编号，**不是排针孔位序号**。本课示例选 GPIO18 接外部 LED，GPIO19 接外部按钮；这些针脚在 STEP 1 只是 HAL 的功能标签。

最少的电路知识：电压是两点的电位差，电流需要回路；3V3/GND 供电与参考地，GPIO 读写信号。调一调电阻，看看为什么 LED 要限流。没有板子也可以先理解这些关系。


In [ ]:
%esp32_board

### 从接线到程序约定

- LED：`GPIO18 → 330 Ω → LED → GND`，HIGH 表示点亮。实际 LED 要注意正负极。
- Button：`GPIO19 → 按钮 → GND`，输入启用上拉（pull-up），松开时为 HIGH，按下时为 LOW。
- 没有上拉/下拉时，断开的输入可能浮空；软件读到的值不可靠。上拉不是让按钮自动被按下，而是为松开状态提供确定电平。

板卡图是连接原理示意；电脑运行的仍是你编译的 C++。它不仿真电流、射频或芯片指令时序。

## 1. 按住亮灯，松开灭灯

运行下面 C++ 格，在下方深色实验台点击 **按下按钮 / 松开按钮**。观察按钮的 HIGH / LOW 和 LED 的 ON / OFF 一起变化；LED 来自本格 C++ 的实际输出。

点击 **采样 +10 ms** 继续调用 `tick()`；打开 **连续采样（慢放）** 后，每 250 ms 墙钟时间推进 10 ms 虚拟时间，便于观察。关闭它可以单步。**播放自动检查** 会动画演示各个操作，首次不符会标红，检查项也会标红。

读图提示：LED 点亮为黄色，熄灭为灰色；同时看 ON / OFF，避免只靠颜色判断。绿色 ✓ 表示当前符合要求；红色 ✗ 会说明预期与实际的差异，并保留首次失败，点击 **复位** 清除。长说明都在图外，会随面板宽度换行。

当前规则：按住亮、松开灭。`%%cpp_lab` 是运行标记，后面整格是正常 C++。


In [ ]:
%%cpp_lab guided guided pass
#pragma once
#include "academy/board.hpp"

namespace academy {
class Controller {
public:
    explicit Controller(Board& board) : board_(board) {}

    void tick() {
        const bool pressed = board_.read_button() == Level::low;
        board_.write_led(pressed);
    }

private:
    Board& board_;

};
} // namespace academy


## 2. 修改一次，马上看见变化

将下面这一格的 `write_led(pressed)` 改成 `write_led(!pressed)`，重新运行。

运行前先预测：松开时 LED 会亮还是灭？点击按钮确认。随后把这格恢复原状再运行。

每格都会重新编译并从初始状态运行，不会继承上一格 C++ 对象的状态。你不需要去磁盘上找源码文件。


In [ ]:
%%cpp_lab modify
#pragma once
#include "academy/board.hpp"

namespace academy {
class Controller {
public:
    explicit Controller(Board& board) : board_(board) {}

    void tick() {
        const bool pressed = board_.read_button() == Level::low;
        board_.write_led(!pressed);
    }

private:
    Board& board_;

};
} // namespace academy


## 3. 故障实验：按一下切换，为什么却闪个不停？

提醒器需要“按一下开启提醒，再按一下关闭”；释放按钮应该保持灯的状态。

下面的实现每次读到 `LOW` 就反转灯。先预测长按期间会发生什么，再运行。测试失败是**预期现象**，不需要修复运行环境。

按下后打开连续采样：LED 为什么反复闪烁？或点击“播放自动检查”，看哪一步标红。一次物理按下是否等于一次采样？


In [ ]:
%%cpp_lab broken exercise fail
#pragma once
#include "academy/board.hpp"

namespace academy {
class Controller {
public:
    explicit Controller(Board& board) : board_(board) {}

    void tick() {
        const bool pressed = board_.read_button() == Level::low;
        if (pressed) {
            led_on_ = !led_on_;
        }
        board_.write_led(led_on_);
    }

private:
    Board& board_;
    bool led_on_ = false;
};
} // namespace academy


## 4. 修复你这一格的 C++

在下面代码里添加需要的状态，并修改 `tick()`。运行后，测试检查的就是**这一格的代码**，不会偷偷换成参考答案。

行为契约：

- 第一次新按下开灯，长按保持亮；释放仍亮。
- 第二次新按下灭灯，第三次重新亮。
- 上电时如果按钮已经按住，先把它当成 baseline；必须释放后再次按下才算一次新按下。
- 两个 Controller 的状态不能互相干扰。

这格保留你此前的实现。运行后先手动操作，再播放检查；修改后重跑本格会关闭旧面板并创建新面板。需要帮助时再打开 [分层 Hint](../lessons/01-smart-button/hints.md)。


In [ ]:
%%cpp_lab exercise exercise
#pragma once
#include "academy/board.hpp"

namespace academy {
class Controller {
public:
    explicit Controller(Board& board) : board_(board) {}

    void tick() {
        const bool pressed = board_.read_button() == Level::low;
        // TODO: 这里应只响应一次新的按下。可以添加成员变量。
        if (!pressed) {
            IsPressed = false;
        }
        if (pressed && !IsPressed) {
            led_on_ = !led_on_;
            IsPressed = true; 
        }
        
        
        board_.write_led(led_on_);
    }

private:
    Board& board_;
    bool led_on_ = false;
    bool IsPressed = true;
};
} // namespace academy


## 5. 解释你的修复

在下面文字格双击编辑，写两三句即可：你保存了什么状态？什么时候才改变 LED？为什么不把状态放在共享的全局变量里？


我的记录：

- 默认已经按下为真，循环开始后判断并更新该IsPressed变量。
- 当松开，被认为结束一次按键动作，IsPressed为假，灯光可以被下一次按键激活
- 按下第个tick，按下激活。IsPressed更新为真，代表本次动作已经响应，无需再次控制灯光。直到下次松开按键。


## 6. 这份 C++ 怎样变成 ESP32 固件？

Notebook 把你这一格 C++ 编译成电脑程序，接入 `VirtualBoard`；将来使用同一个 `Board` 合约，改接 `Esp32Board`。

`Controller → Board → Virtual GPIO / ESP32 GPIO`

你刚写的事件逻辑可以保留。真实板子还需要配置输入、接线、检查极性和按键抖动。我们还没有解决 debounce；快速抖动会产生多个边沿，是下一轮实验的问题。

**开放挑战**：先写出你认为 debounce 应当保证的行为，再决定如何引入时间。

完成尝试后才查看 [独立 Solution Notebook](solutions/01-smart-button-solution.ipynb)。先不扩展课程；请反馈哪个代码格或哪个输出帮助了你，哪个步骤仍然不直观。
